# 8차시 학생용 실습 — 공정 이상과 원인 후보 찾기

오늘의 질문: **온도가 이상했던 시점과 불량이 발생한 시점이 겹칠까?**

위에서부터 한 셀씩 차례로 실행하세요(Shift+Enter). `# TODO` 표시가 있는 셀은 `____` 빈칸을 채운 뒤 실행하고, 나머지 셀은 실행하며 결과를 확인하면 됩니다.

## 1단계. 데이터 불러오기

### 따라 해보기 — 팔굽혀펴기 횟수로 관리 상한/하한 감 잡기
강의 「이상치, 관리 상한/하한, 표준편차」의 작은 예제입니다. 친구 6명의 팔굽혀펴기 횟수로 **평균 ± 2×표준편차** 경계선을 계산해 봅니다. 실제 공정 관리도는 보통 3배를 쓰지만, 이 과정에서는 이상치가 잘 보이도록 교육용으로 2배를 씁니다.

In [ ]:
# 친구 6명의 팔굽혀펴기 횟수를 pd.Series(숫자를 한 줄로 담는 표)로 만듭니다.
# 실행하면 왼쪽에 번호(0~5), 오른쪽에 횟수가 보입니다 — 한 명만 유독 큰 값인지 확인해 보세요.
import pandas as pd

pushups = pd.Series([20, 22, 19, 21, 20, 85])   # 친구 6명의 팔굽혀펴기 횟수
pushups  # (중간 확인)

In [ ]:
# 평균(mean)과 표준편차(std)를 구하고, 평균 ± 2×표준편차로 관리 상한(ucl)·관리 하한(lcl)을 계산합니다.
# 실행하면 "평균·표준편차"와 "관리 상한·관리 하한"이 두 줄로 출력됩니다.
# .std(): 값들이 평균에서 얼마나 흩어져 있는지(표준편차)를 계산합니다.
mean = pushups.mean()
std = pushups.std()
ucl = mean + 2 * std
lcl = mean - 2 * std

# f"...{값:.2f}...": 문자열 안에 값을 소수 둘째 자리까지 넣어 출력합니다.
print(f"평균: {mean:.2f}, 표준편차: {std:.2f}")
print(f"관리 상한: {ucl:.2f}, 관리 하한: {lcl:.2f}")

In [ ]:
# 관리 상한보다 크거나(|) 관리 하한보다 작은 값만 골라 냅니다.
# 실행하면 이상치로 잡힌 값의 번호와 횟수가 출력됩니다.
print(pushups[(pushups > ucl) | (pushups < lcl)])

### 오늘의 데이터 불러오기
이제 오늘 다룰 반도체 공정 데이터(`week08_anomaly_root_cause.csv`, 가상 데이터)를 불러옵니다.

In [ ]:
# 오늘 사용할 CSV 파일을 읽어 df라는 표에 저장합니다.
# 실행하면 표의 맨 앞 5행이 보입니다 — 측정시간부터 합격여부까지 열 12개를 훑어보세요.
import pandas as pd

df = pd.read_csv("../../data/weekly/week08/week08_anomaly_root_cause.csv")
df.head()  # (중간 확인)

In [ ]:
# 표의 크기를 (행 개수, 열 개수)로 확인합니다.
# 실행하면 괄호 안에 숫자 두 개가 출력됩니다 — 앞은 측정 기록(행) 수, 뒤는 열 수입니다.
print(df.shape)  # (중간 확인)

## 2단계. 진공도 정상 범위 이탈 데이터 찾기
정상 범위: 4.5 ~ 5.5 mTorr

In [ ]:
# 진공도의 정상 범위(4.5~5.5 mTorr)를 변수로 정하고, 범위를 벗어난 행만 골라 vac_outlier에 저장합니다.
# 실행하면 이탈한 행의 앞 5개가 보입니다 — 진공도_mTorr 값이 4.5보다 작거나 5.5보다 큰지 확인해 보세요.
# 1. 정상 범위 경계값을 변수로 정한다.
vac_low, vac_high = 4.5, 5.5

# 2. 조건 검색으로 정상 범위를 벗어난 행을 찾는다.
# (조건1) | (조건2): |는 "또는(OR)" — 둘 중 하나만 맞아도 선택됩니다. 각 조건은 괄호로 감쌉니다.
# TODO: 빈칸을 채워 정상 범위를 벗어난 행을 찾으세요. (미만 또는 초과 조건을 '|'로 연결)
vac_outlier = df[(df["진공도_mTorr"] < vac_low) ____ (df["진공도_mTorr"] > vac_high)]
vac_outlier.head()  # (중간 확인)

In [ ]:
# 전체 행 수, 진공도 이탈 행 수, 이탈 비율(%)을 한 줄씩 출력합니다.
# len(표)는 행 개수를 세고, round(값, 2)는 소수 둘째 자리까지 반올림합니다.
# 실행하면 세 줄이 보입니다 — 전체 중 몇 %가 정상 범위를 벗어났는지 확인해 보세요.
print("전체 행 수:", len(df))
print("진공도 이탈 행 수:", len(vac_outlier))
print("이탈 비율(%):", round(len(vac_outlier) / len(df) * 100, 2))

## 3단계. 진공도 이탈 여부에 따른 불합격률 비교

In [ ]:
# 이번에는 반대로, 진공도가 정상 범위 안(4.5 이상 5.5 이하)인 행만 골라 vac_in에 저장합니다.
# &는 "그리고(AND)" — 두 조건이 모두 맞아야 선택됩니다. 실행하면 범위 안 행 개수가 출력됩니다.
vac_in = df[(df["진공도_mTorr"] >= vac_low) & (df["진공도_mTorr"] <= vac_high)]
print("정상 범위 안 행 수:", len(vac_in))  # (중간 확인)

In [ ]:
# 합격여부가 -1(불합격)인 비율을 범위 안/밖 각각 계산해 %로 바꿉니다.
# (조건).mean(): True를 1, False를 0으로 보고 평균을 내므로 "조건에 맞는 비율"이 됩니다.
# 실행하면 두 불합격률이 출력됩니다 — 범위 밖이 몇 배쯤 높은지 비교해 보세요.
fail_rate_in = (vac_in["합격여부"] == -1).mean() * 100
fail_rate_out = (vac_outlier["합격여부"] == -1).mean() * 100

print("정상 범위 안 불합격률(%):", round(fail_rate_in, 2))
print("정상 범위 밖 불합격률(%):", round(fail_rate_out, 2))

**질문**: 두 불합격률 차이를 보고, '원인 후보'라는 표현을 사용해 한 문장으로 정리해보세요.

## 4단계. 온도와 두께의 관계(산점도 + 상관계수)

### 따라 해보기 — 공부 시간과 시험 점수로 corr() 감 잡기
6차시에서 그려본 "공부 시간과 시험 점수" 예로, `.corr()`가 상관계수를 어떻게 계산해주는지 먼저 확인합니다.

In [ ]:
# 공부 시간과 시험 점수 12쌍을 만들고, 두 목록의 상관계수를 계산합니다.
# A.corr(B): A와 B가 함께 움직이는 정도를 -1 ~ 1 사이 숫자(상관계수 r)로 알려 줍니다.
# 실행하면 r이 출력됩니다 — 1에 가까우면 "한쪽이 클수록 다른 쪽도 크다"는 강한 양의 상관입니다.
study_hours = pd.Series([1, 2, 2, 3, 4, 4, 5, 6, 6, 7, 8, 8])
exam_scores = pd.Series([50, 65, 55, 68, 70, 80, 72, 88, 78, 85, 90, 80])

r_study = study_hours.corr(exam_scores)
print("공부시간-점수 상관계수 r =", round(r_study, 2))

### 반도체 데이터에 적용하기 — 온도와 두께
같은 `.corr()`를 온도_섭씨 · 두께_nm 두 숫자 열에 적용하고, 산점도로 모양도 확인합니다.

In [ ]:
# 그래프 도구(matplotlib)를 plt라는 이름으로 불러오고(다음 셀에서 사용), 그래프에 한글이 제대로 보이도록 글꼴을 정합니다.
# 그다음 온도_섭씨와 두께_nm의 상관계수를 r에 저장합니다. 실행하면 상관계수 r이 출력됩니다 — 위 공부시간 예의 r과 비교해 보세요.
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 글꼴(Windows). Mac은 "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False      # 음수 부호(-)가 네모로 깨지지 않게

# TODO: 빈칸에 상관계수를 계산하는 메서드 이름을 채우세요.
r = df["온도_섭씨"].____(df["두께_nm"])
print("상관계수 r =", round(r, 2))

In [ ]:
# 가로축에 온도, 세로축에 두께를 놓고 점을 찍는 산점도를 그립니다(alpha=0.6: 점을 조금 투명하게 해 겹친 곳이 진하게 보임).
# 실행하면 그래프가 나타납니다 — 점들이 오른쪽 위로 올라가는 모양인지 보세요. 제목의 r은 바로 위 셀에서 계산한 값입니다.
plt.scatter(df["온도_섭씨"], df["두께_nm"], alpha=0.6)
plt.xlabel("온도_섭씨")
plt.ylabel("두께_nm")
plt.title(f"온도와 두께의 관계 (r = {r:.2f})")
plt.show()

## 5단계. 습도(대조군)와 불합격의 관계 확인하기
습도는 처음부터 불합격과 무관하게 설계된 '대조군' 변수입니다. 정말 그런지 확인해봅시다.

In [ ]:
# 합격여부(1 = 합격, -1 = 불합격)를 계산하기 쉬운 숫자(불합격 1, 합격 0)로 바꾼 새 열을 만듭니다.
# .astype(int): True/False를 1/0으로 바꿉니다. 실행하면 두 열이 나란히 보입니다 — -1인 행이 1로 바뀌었는지 확인해 보세요.
df["불합격여부_숫자"] = (df["합격여부"] == -1).astype(int)  # 불합격이면 1, 합격이면 0
df[["합격여부", "불합격여부_숫자"]].head()  # (중간 확인)

In [ ]:
# 습도_pct와 불합격여부_숫자의 상관계수를 계산합니다.
# 실행하면 상관계수 r이 출력됩니다 — 0에 가까울수록 두 값이 한 방향으로 함께 움직이지 않는다는 뜻입니다.
r_humid = df["습도_pct"].corr(df["불합격여부_숫자"])
print("습도-불합격 상관계수 r =", round(r_humid, 2))

**질문**: 습도의 상관계수는 온도-두께의 상관계수보다 훨씬 0에 가깝습니다.
이 결과가 '분석 실패'가 아니라 '의미 있는 발견'인 이유는 무엇일까요?

## 6단계(도전). EQ-02의 진동 값 비교하기

In [ ]:
# 설비번호별로 묶어(groupby) 진동_mm_s의 평균을 구하고, 소수 셋째 자리까지 반올림합니다.
# 실행하면 설비 4대의 평균 진동이 출력됩니다 — EQ-02가 다른 설비보다 높은지 확인해 보세요.
# TODO: 설비별 진동_mm_s 평균을 구해 EQ-02가 다른 설비보다 높은지 확인해보세요. (빈칸: 평균을 구하는 메서드)
print(df.groupby("설비번호")["진동_mm_s"].____().round(3))

### 냉각수온도가 오르면 공정 온도가 뒤따라 오를까? — 이상 전후 비교
강의 「실습 코드 ④」의 두 번째 예제입니다(과제가 아니라 실행하며 결과를 확인하는 셀입니다). 전체 데이터의 냉각수온도-공정온도 상관계수는 약 0.06으로 거의 0이지만, 냉각수온도가 오르기 시작한 구간 하나를 골라 "오르기 전 10건"과 "오르는 동안 10건"을 직접 비교해 봅니다.

`df.iloc[시작:끝]`은 조건이 아니라 **순서(위치)**로 행을 고릅니다 — 시작 번호는 포함, 끝 번호는 포함하지 않습니다(4차시). 데이터가 측정시간 순으로 정렬되어 있어야 뜻대로 동작합니다.

In [ ]:
# iloc으로 321~330번 행(냉각수온도가 오르기 전 10건)과 331~340번 행(오르는 동안 10건)을 잘라 냅니다.
# 실행하면 두 구간의 냉각수온도 평균이 출력됩니다 — 얼마나 올랐는지 확인해 보세요.
# 데이터가 이미 측정시간 순으로 저장되어 있으므로, 행 번호로 구간을 나눌 수 있다.
before = df.iloc[321:331]   # 냉각수온도가 오르기 전 10건
during = df.iloc[331:341]   # 냉각수온도가 오르는 구간 10건

print("이전 냉각수온도 평균:", round(before["냉각수온도_섭씨"].mean(), 2))
print("이후 냉각수온도 평균:", round(during["냉각수온도_섭씨"].mean(), 2))

In [ ]:
# 같은 두 구간에서 공정온도(온도_섭씨)의 평균을 비교합니다.
# 실행하면 두 평균이 출력됩니다 — 냉각수온도가 오르는 동안 공정온도도 함께 올랐는지 확인해 보세요.
print("이전 공정온도 평균:", round(before["온도_섭씨"].mean(), 2))
print("이후 공정온도 평균:", round(during["온도_섭씨"].mean(), 2))

## 7단계. 오늘의 분석을 한 문장으로 정리하기

> (예시) 진공도가 정상 범위를 벗어나면 불합격률이 ___%에서 ___%로 높아지는 경향이 관찰되어,
> 진공도 이탈은 불합격의 원인 후보로 볼 수 있다. 다만 상관관계만으로 원인이라고 단정할 수 없다.

### 도전 문제 1 — 압력_Pa도 정상 범위 안/밖 불합격률 비교 (빠른 학습자용)
강의 「도전 실습」의 도전 문제입니다(실습지 7번과 같은 과제). 압력_Pa 열에도 진공도와 같은 방식(정상 범위 밖 vs 안)으로 불합격률을 비교합니다. 교육용 정상 범위는 **995~1025 Pa**(7차시와 동일)입니다. 2단계·3단계에서 진공도에 쓴 코드를 떠올리며 풀고, 진공도만큼 차이가 크게 나는지 확인해 보세요.

In [ ]:
# 압력_Pa의 교육용 정상 범위(995~1025 Pa)를 변수로 정하고, 범위 밖(pres_outlier)과 범위 안(pres_in) 행을 각각 골라 냅니다.
# 2단계(진공도 범위 밖)·3단계(진공도 범위 안)와 같은 모양의 조건식입니다. 실행하면 두 집단의 행 수가 출력됩니다.
pres_low, pres_high = 995, 1025

# TODO: 두 빈칸에 조건과 조건을 잇는 기호를 채우세요. (범위 밖: 둘 중 하나만 맞아도 선택 / 범위 안: 둘 다 맞아야 선택)
pres_outlier = df[(df["압력_Pa"] < pres_low) ____ (df["압력_Pa"] > pres_high)]
pres_in = df[(df["압력_Pa"] >= pres_low) ____ (df["압력_Pa"] <= pres_high)]

print("압력 이탈 행 수:", len(pres_outlier))
print("압력 정상 범위 안 행 수:", len(pres_in))

> **참고**
> - 첫 번째 빈칸(범위 **밖**): 강의 「② 파이썬으로 확인하기 — 정상 범위 설정과 이탈 데이터 찾기」의 `vac_outlier` 줄과 "코드 한 줄씩 설명"을 다시 보세요. 아래 조건 **또는** 위 조건 중 하나만 맞아도 골라야 하므로 "또는(OR)"을 뜻하는 기호가 들어갑니다.
> - 두 번째 빈칸(범위 **안**): 강의 「실습 코드 ② — 진공도 이탈과 불합격률 비교」의 `vac_in` 줄을 다시 보세요. 두 조건이 **모두** 맞아야 하므로 "그리고(AND)"를 뜻하는 기호가 들어갑니다. 각 조건을 괄호로 감싸는 것도 잊지 마세요.

In [ ]:
# 압력 범위 안/밖 각각에서 합격여부가 -1(불합격)인 비율을 %로 계산합니다(3단계 진공도 계산과 같은 방법).
# 실행하면 두 불합격률이 출력됩니다 — 3단계 진공도(범위 안 8.06, 범위 밖 25.64)만큼 차이가 나는지 비교해 보세요.
# TODO: 두 빈칸에 "조건에 맞는 행의 비율"을 구하는 메서드 이름을 채우세요(두 빈칸은 같은 메서드).
pres_fail_in = (pres_in["합격여부"] == -1).____() * 100
pres_fail_out = (pres_outlier["합격여부"] == -1).____() * 100

print("압력 정상 범위 안 불합격률(%):", round(pres_fail_in, 2))
print("압력 정상 범위 밖 불합격률(%):", round(pres_fail_out, 2))

> **참고**
> - 두 빈칸(같은 메서드): 강의 「실습 코드 ② — 진공도 이탈과 불합격률 비교」의 `fail_rate_in`·`fail_rate_out` 줄을 다시 보세요. `(조건)` 뒤에 이 메서드를 붙이면 True를 1, False를 0으로 보고 평균을 내므로 "조건에 맞는 행의 비율"이 되고, `* 100`을 곱하면 %가 됩니다. `합격여부 == -1`은 불합격이라는 뜻입니다(1 = 합격, -1 = 불합격).

**질문**: 압력 범위 밖의 불합격률은 3단계의 진공도(8.06% → 25.64%)만큼 높아졌나요? 압력을 불합격의 원인 후보로 볼 수 있을지, 근거 수치를 넣어 자신의 문장으로 적어 보세요.

## 8단계. AI에게 질문하며 더 알아보기
오늘 배운 상관관계·원인 후보 개념에 대해 AI 챗봇에게 질문해서 이해를 넓혀봅시다. 하나 이상
골라 물어보고, 새로 알게 된 내용을 다음 셀에 한두 줄로 정리해보세요.

- "상관계수(r)가 0.7이면 '강한 상관관계'라고 할 수 있나요? 판단 기준이 있나요?"
- "상관관계와 인과관계는 정확히 어떻게 다른가요?"
- "'대조군(control group)'이라는 개념이 왜 중요한가요?"
- "여러 변수 중 어떤 변수가 불량과 가장 관련 있는지 한 번에 비교하려면 어떻게 하나요?"

✏️ **내가 새로 알게 된 점**: (여기에 AI 답변을 요약해서 적어보세요)

## 9단계. AI에게 코드 생성 요청하고 직접 실행해보기
아래 프롬프트를 AI에게 그대로 물어보고, AI가 만들어준 코드를 다음 셀에 붙여넣어 실행해보세요.

- 프롬프트 예시: "두 숫자 리스트의 상관계수를 계산하고, 절댓값이 0.5보다 크면 '뚜렷한 관계',
  아니면 '약한 관계'라고 출력하는 코드를 만들어줘."

In [ ]:
# AI가 만들어 준 코드를 이 셀에 붙여넣고 실행합니다.
# 실행하면 AI 코드의 결과(상관계수와 판정)가 이 셀 아래에 출력됩니다.
# 여기에 AI가 생성한 코드를 붙여넣고 실행해보세요.

## 10단계. 연습문제 — 진동 값과 불합격의 관계 확인하기
6단계에서 설비별 진동_mm_s 평균만 확인해봤습니다. 이번에는 진동_mm_s과 불합격여부_숫자
사이의 상관계수까지 직접 계산해, 진동이 불합격의 원인 후보가 될 수 있는지 살펴봅시다.

In [ ]:
# 진동_mm_s와 불합격여부_숫자(5단계에서 만든 열)의 상관계수를 계산합니다.
# 실행하면 상관계수 r이 출력됩니다 — 온도-두께(약 0.71)와 습도-불합격(약 -0.13) 중 어느 쪽에 가까운지 비교해 보세요.
# TODO: 진동_mm_s과 불합격여부_숫자의 상관계수를 계산하세요.
r_vib = df["진동_mm_s"].____(df["불합격여부_숫자"])
print("진동-불합격 상관계수 r =", round(r_vib, 2))

## 11단계. AI로 재미있는 미니 프로그램 만들기 🎉 — 원인 후보 추리 게임
오늘 계산한 상관계수들(온도-두께, 습도-불합격 `r_humid`, 진동-불합격 `r_vib`)을 한데 모아, 절댓값이 큰 순서로 "용의선상 원인 후보
랭킹"을 매겨주는 재미있는 프로그램을 AI에게 만들어달라고 요청해봅시다. 9단계 코드에서 `r`을 다른 값으로 덮어썼다면, 온도-두께 상관계수는 `r_thickness`로 다시 계산해서 쓰세요.

**프롬프트 예시**: "변수 이름과 상관계수가 담긴 딕셔너리를 받아서, 절댓값이 큰 순서로 정렬해
'용의선상 원인 후보 랭킹'처럼 🔎 이모지와 함께 출력하는 파이썬 코드를 만들어줘."

**더 놀아보기**:
- 다른 변수 쌍의 상관계수도 추가해서 더 긴 랭킹을 만들어보거나, 1위 후보에게만 특별한 문구를
  붙여달라고 요청해보세요.

In [ ]:
# AI가 만들어 준 원인 후보 추리 게임 코드를 이 셀에 붙여넣고 실행합니다.
# 실행하면 상관계수가 큰 순서로 매긴 순위가 출력되는지 확인해 보세요(출력 문장에도 "원인 후보"라고 쓰는지 보세요).
# 여기에 AI가 생성한 원인 후보 추리 게임 코드를 붙여넣고 실행해보세요.